# Day 033 Project: Async Batch Classifier

## What You're Building

A `BatchProcessor` that classifies a list of sentences concurrently — demonstrating that async batch processing is faster than serial for I/O-bound LLM calls.

## Project Requirements

1. Define a list of at least 8 sentences across two or more categories    (e.g., positive/negative, question/statement, short/long)
2. Create a `BatchProcessor` with `max_concurrent=3`
3. Use `await bp.process(items, prompt_fn)` to classify all items concurrently
4. Print a summary showing how many items were classified in each category
5. Time the batch and compare to an estimate of serial time
6. Verify with `_run_project_checks()`

## Provided: All Helper Functions + BatchProcessor

In [ ]:
import asyncio
import ollama

async def async_chat(prompt: str, model: str = "llama3.2") -> str:
    client = ollama.AsyncClient()
    response = await client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]


import asyncio

async def gather_results(coros: list) -> list:
    return list(await asyncio.gather(*coros))


async def throttled_gather(coros: list, max_concurrent: int) -> list:
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(coro):
        async with sem:
            return await coro
    return list(await asyncio.gather(*[_run(c) for c in coros]))


async def process_batch(
    items: list, async_fn, max_concurrent: int = 3
) -> list[dict]:
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(item):
        async with sem:
            try:
                result = await async_fn(item)
                return {"item": item, "status": "ok",
                        "result": result, "error": None}
            except Exception as e:
                return {"item": item, "status": "error",
                        "result": None, "error": str(e)}
    return list(await asyncio.gather(*[_run(i) for i in items]))


class BatchProcessor:
    def __init__(self, max_concurrent: int = 3, model: str = "llama3.2"):
        self.max_concurrent = max_concurrent
        self.model          = model

    async def process(self, items: list, prompt_fn) -> list[dict]:
        async def _call(item):
            return await async_chat(prompt_fn(item), self.model)
        return await process_batch(items, _call, self.max_concurrent)

    def run(self, items: list, prompt_fn) -> list[dict]:
        return asyncio.run(self.process(items, prompt_fn))

## Your Batch Classification

In [ ]:
import time

# At least 8 sentences to classify
SENTENCES = [
    'I absolutely love this product!',
    'This is the worst experience I have ever had.',
    'It was okay, nothing special.',
    'Fantastic quality, would buy again.',
    'Total waste of money.',
    'Pretty good, met my expectations.',
    'Exceeded all my expectations!',
    'Not impressed at all.',
]

def classify_prompt(sentence: str) -> str:
    return (
        f"Classify this review as 'positive', 'negative', or 'neutral'. "
        f"Reply with one word only.\n\nReview: {sentence}"
    )

bp = BatchProcessor(max_concurrent=3)

# TODO: await bp.process(SENTENCES, classify_prompt) and store as results
# TODO: print results and summary
# TODO: time the batch and print elapsed seconds


## Checks

In [ ]:
async def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: bp is a BatchProcessor
    try:
        assert 'bp' in globals()
        assert isinstance(bp, BatchProcessor), \
            f'bp should be BatchProcessor, got {type(bp)}'
        passed += 1; print('\u2705 Check 1: bp is a BatchProcessor')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: SENTENCES has ≥ 8 items
    try:
        assert 'SENTENCES' in globals()
        assert len(SENTENCES) >= 8, \
            f'need ≥8 sentences, got {len(SENTENCES)}'
        passed += 1; print(f'\u2705 Check 2: {len(SENTENCES)} sentences defined')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: results defined and has correct length
    try:
        assert 'results' in globals()
        assert len(results) == len(SENTENCES), \
            f'results length {len(results)} != sentences length {len(SENTENCES)}'
        passed += 1; print(f'\u2705 Check 3: {len(results)} results for {len(SENTENCES)} sentences')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: all results are dicts with required keys
    try:
        for r in results:
            for k in ('item', 'status', 'result', 'error'):
                assert k in r, f'missing key {k!r}: {r}'
        ok = [r for r in results if r['status'] == 'ok']
        assert len(ok) > 0, 'no successful classifications'
        passed += 1; print(f'\u2705 Check 4: {len(ok)}/{len(results)} items classified successfully')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: bp.max_concurrent == 3
    try:
        assert bp.max_concurrent == 3, \
            f'max_concurrent should be 3, got {bp.max_concurrent}'
        passed += 1; print('\u2705 Check 5: max_concurrent=3 set correctly')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


await _run_project_checks()

## Bonus Challenges

- Add timing: measure total batch time and compare to `len(SENTENCES) * estimate_per_call`
  to show the speedup factor
- Process two different datasets concurrently using `gather_results` of two   `BatchProcessor.process()` calls at the same time
- Add retry logic: after the batch, re-run `process_batch` on the error items   from the first run (combine with Day 031 retry)
- Try `max_concurrent` values of 1, 3, 5 and plot (print) the wall time for each —
  at what point does more concurrency stop helping?
- Integrate `SecureConfig` (Day 032) to load the model name and max_concurrent   from a `.env` string instead of hardcoding them